In [1]:
# launch Isaac Sim first (before any other isaaclab import). One env per kernel: restart to switch tasks.
from api import sim

app = sim.launch(cameras=True)

[INFO][AppLauncher]: Using device: cuda:0
[INFO][AppLauncher]: Loading experience file: /scratch/pranamlab/farazr/control-proj/control/deps/IsaacLab/apps/isaaclab.python.headless.rendering.kit
[Warning] [simulation_app.simulation_app] fast shutdown not supported with jupyter notebooks
Loading user config located at: '/scratch/pranamlab/farazr/control-proj/control/.venv/lib/python3.11/site-packages/isaacsim/kit/data/Kit/Isaac-Sim/5.1/user.config.json'
[Info] [carb] Logging to file: /scratch/pranamlab/farazr/control-proj/control/.venv/lib/python3.11/site-packages/isaacsim/kit/logs/Kit/Isaac-Sim/5.1/kit_20261002_204520.log
2026-10-03T00:45:20Z s] [Warning] [carb.windowing-glfw.plugin] GLFW initialization failed.
2026-10-03T00:45:20Z s] [Warning] [carb] Failed to startup plugin carb.windowing-glfw.plugin (interfaces: [carb::windowing::IGLContext v1.0],[carb::windowing::IWindowing v1.5]) (impl: carb.windowing-glfw.plugin)
2026-10-03T00:45:20Z s] [Warning] [omni.platforminfo.plugin] failed t

2026-10-03T00:45:28Z [8,773ms] [Error] [asyncio] Exception in callback _patch_task.<locals>.step()
handle: <Handle _patch_task.<locals>.step()>
Traceback (most recent call last):
  File "/mnt/lcars/home/f/farazr/.local/share/uv/python/cpython-3.11.16-linux-x86_64-gnu/lib/python3.11/asyncio/events.py", line 84, in _run
    self._context.run(self._callback, *self._args)
RuntimeError: cannot enter context: <_contextvars.Context object at 0x72b6f82360c0> is already entered
2026-10-03T00:45:34Z [13,950ms] [Error] [asyncio] Task was destroyed but it is pending!
task: <Task pending name='Task-8' coro=<_async_in_context.<locals>.run_in_context() done, defined at /scratch/pranamlab/farazr/control-proj/control/.venv/lib/python3.11/site-packages/ipykernel/utils.py:57> wait_for=<Task pending name='Task-9' coro=<Kernel.shell_main() running at /scratch/pranamlab/farazr/control-proj/control/.venv/lib/python3.11/site-packages/ipykernel/kernelbase.py:597> cb=[Task.__wakeup()]> cb=[ZMQStream._run_callba

2026-10-03T00:45:30Z [10,120ms] [Warning] [carb.graphics-vulkan.plugin] NGX DLSS Frame Generation Feature AdapterUnsupported
2026-10-03T00:45:33Z [13,176ms] [Warning] [omni.fabric.plugin] Warning: attribute viewportHandle not found for bucket id 7

2026-10-03T00:45:33Z [13,306ms] [Warning] [rtx.postprocessing.plugin] DLSS-RR is not supported in the current build due to some known bugs with this GPU: NVIDIA H100 NVL.Note that RTX Real-Time raytracing mode will not be able to denoise the output correctly.
[INFO]: Parsing configuration from: isaaclab_tasks.direct.shadow_hand.shadow_hand_env_cfg:ShadowHandEnvCfg

[INFO][IsaacLab]: Logging to file: /tmp/isaaclab/logs/isaaclab_2026-10-02_20-45-34.log

20:45:34 [simulation_context.py] WARNING: The `enable_external_forces_every_iteration` parameter in the PhysxCfg is set to False. If you are experiencing noisy velocities, consider enabling this flag. You may need to slightly increase the number of velocity iterations (setting it to 1 or 2 rath

In [2]:
import torch

TASK = "Isaac-Repose-Cube-Shadow-Direct-v0"


def closeup_camera(cfg):
    # close-up camera on env 0 (default views the whole grid from far away)
    cfg.viewer.origin_type = "env"
    cfg.viewer.eye = (0.45, -0.85, 0.95)
    cfg.viewer.lookat = (0.0, -0.35, 0.55)


env = sim.make_env(TASK, num_envs=1, render=True, cfg_fn=closeup_camera)

2026-10-03T00:45:36Z [16,067ms] [Error] [isaacsim.core.cloner.impl.cloner] Failed to clone in Fabric


In [3]:
# random actions; swap in a policy here
obs, _ = env.reset()
frames = []
for _ in range(300):
    action = 2 * torch.rand(env.action_space.shape, device=env.unwrapped.device) - 1
    obs, reward, terminated, truncated, info = env.step(action)
    frames.append(env.render())

In [4]:
from api.video import show

fps = round(1 / env.unwrapped.step_dt)  # real time
show(frames, fps=fps)

In [5]:
# pretrained rl_games policy (NVIDIA-published checkpoint), loaded as a plain torch module
from api import policies

policy = policies.load_rl_games(TASK, device=env.unwrapped.device)

obs, _ = env.reset()
frames = []
for _ in range(600):
    obs, reward, terminated, truncated, info = env.step(policy(obs))
    frames.append(env.render())

In [6]:
show(frames, fps=fps)